# 12 端侧评估指标体系（可运行脚手架）

把 TTFT、ITL、吞吐、内存、精度、功耗收成一套可复用的评测函数，方便对比量化/框架/硬件。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

torch.manual_seed(42)
np.random.seed(42)
print(f"PyTorch {torch.__version__} | CUDA={torch.cuda.is_available()}")

import time
from dataclasses import dataclass, asdict

## 12.1 延迟与吞吐

In [ ]:
@dataclass
class LatencyReport:
    ttft_ms: float
    itl_ms: float
    e2e_ms: float
    tokens: int

    @property
    def tok_per_s(self) -> float:
        gen_ms = max(self.e2e_ms - self.ttft_ms, 1e-6)
        return (self.tokens - 1) / (gen_ms / 1000) if self.tokens > 1 else 0.0


def simulate_generate(n_tokens=32, ttft_ms=120.0, itl_ms=28.0, jitter=0.1):
    """用睡眠模拟 decode，便于本地无模型时跑通评测管线。"""
    t0 = time.perf_counter()
    time.sleep(ttft_ms / 1000 * (1 + np.random.uniform(-jitter, jitter)))
    ttft = (time.perf_counter() - t0) * 1000
    itls = []
    for _ in range(n_tokens - 1):
        t1 = time.perf_counter()
        time.sleep(itl_ms / 1000 * (1 + np.random.uniform(-jitter, jitter)))
        itls.append((time.perf_counter() - t1) * 1000)
    e2e = (time.perf_counter() - t0) * 1000
    return LatencyReport(ttft, float(np.mean(itls)), e2e, n_tokens)


rep = simulate_generate()
print("=== 延迟报告 ===")
print(asdict(rep))
print(f"吞吐: {rep.tok_per_s:.1f} tok/s")

## 12.2 资源指标：峰值内存 / 带宽利用率（示意）

In [ ]:
def estimate_llm_memory_mb(params_b, bits=4, kv_tokens=2048, layers=28, kv_heads=8, head_dim=128, kv_bits=16):
    weight = params_b * 1e9 * bits / 8 / 1e6
    # 双向 KV
    kv = 2 * layers * kv_tokens * kv_heads * head_dim * (kv_bits/8) / 1e6
    return {"weight_mb": weight, "kv_mb": kv, "total_mb": weight + kv}


for bits in (16, 8, 4, 2):
    m = estimate_llm_memory_mb(1.5, bits=bits)
    print(f"W{bits}: weight={m['weight_mb']:.0f}MB  KV@2k={m['kv_mb']:.0f}MB  total≈{m['total_mb']:.0f}MB")

## 12.3 精度：PPL 代理与任务分

In [ ]:
def fake_ppl(logits: torch.Tensor, targets: torch.Tensor) -> float:
    loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
    return float(torch.exp(loss))


vocab, seq = 1000, 64
logits = torch.randn(2, seq, vocab)
# 故意构造“较好”分布
logits.scatter_(-1, torch.randint(0, vocab, (2, seq, 1)), 5.0)
targets = logits.argmax(dim=-1)
print(f"代理 PPL: {fake_ppl(logits, targets):.2f}")
print("产业建议: 端到端 PPL 增幅 <0.5；下游任务用同类中文/英文基准。")

## 12.4 一键对比表（量化方案）

In [ ]:
def score_config(name, ttft, tps, mem_mb, ppl_delta, power_w):
    # 简单加权：延迟/内存/功耗越小越好，吞吐越大越好，ppl_delta 越小越好
    score = (50 / ttft) + (tps / 10) + (2000 / mem_mb) + max(0, 2 - ppl_delta) + (5 / power_w)
    return {"name": name, "score": round(score, 2), "ttft": ttft, "tps": tps, "mem": mem_mb, "pplΔ": ppl_delta, "W": power_w}


rows = [
    score_config("FP16", 180, 12, 3200, 0.0, 4.5),
    score_config("W8A16", 140, 18, 1800, 0.1, 3.2),
    score_config("W4A16", 110, 28, 1100, 0.3, 2.4),
    score_config("W2A16", 95, 34, 750, 0.9, 2.0),
]
print(f"{'方案':8s} {'综合分':>6} {'TTFT':>6} {'tok/s':>6} {'MemMB':>7} {'PPLΔ':>5} {'W':>4}")
for r in sorted(rows, key=lambda x: -x["score"]):
    print(f"{r['name']:8s} {r['score']:6.2f} {r['ttft']:6.0f} {r['tps']:6.0f} {r['mem']:7.0f} {r['pplΔ']:5.1f} {r['W']:4.1f}")

## 小结

端侧验收至少同时看：**TTFT、ITL/吞吐、峰值内存、任务精度、持续功耗/热节流**。单看吞吐会误判。